In [ ]:
# === Cài đặt và chuẩn bị ===
!pip install -q geopandas shapely pandas numpy
!git clone https://github.com/lqtue/environmental-data-hub.git 2>/dev/null || echo "Repo đã tồn tại"
%cd environmental-data-hub

# Kiểm tra import
import geopandas, shapely, pandas, numpy
print("Sẵn sàng!")

# Phân tích bão đang hoạt động

Notebook này giúp phân tích nhanh một cơn bão đang hoạt động từ dữ liệu JTWC.

## Nguồn dữ liệu

Tải file KMZ từ **JTWC** (Trung tâm Cảnh báo Bão Liên hợp của Hải quân Mỹ):

- Trang chủ: [https://www.metoc.navy.mil/jtwc/jtwc.html](https://www.metoc.navy.mil/jtwc/jtwc.html)
- Link tải KMZ theo mẫu: `https://www.metoc.navy.mil/jtwc/products/wp{NN}{YYYY}.kmz`
  - `NN` = số thứ tự bão (ví dụ: `01`, `12`)
  - `YYYY` = năm (ví dụ: `2025`)

## Các bước thực hiện

1. **Tải KMZ lên** — chạy ô bên dưới và chọn file KMZ
2. **Chạy các ô tiếp theo** — nhấn nút Play hoặc `Shift+Enter` lần lượt
3. **Xem kết quả** — thông tin bão, so sánh lịch sử, bản đồ

In [ ]:
# === Tải file KMZ lên ===
from google.colab import files
uploaded = files.upload()
kmz_filename = list(uploaded.keys())[0]
print(f"Đã tải lên: {kmz_filename}")

In [ ]:
# === Chuyển đổi KMZ sang GeoJSON ===
from pathlib import Path
from typhoon.kmz_to_geojson import convert

kmz_path = Path(kmz_filename)
geojson_path = Path("data") / f"{kmz_path.stem}.geojson"
convert(kmz_path, geojson_path)
print(f"\nFile GeoJSON: {geojson_path}")

## Đặt tên bão (tuỳ chọn)

Nếu muốn dùng tên tiếng Việt thay vì tên quốc tế, hãy sửa ở ô dưới.

Ví dụ: `"Bão số 7 (YAGI)"`, `"Bão số 3 (GAEMI)"`

Nếu để trống, notebook sẽ dùng tên quốc tế từ file KMZ.

In [ ]:
# Sửa tên bão ở đây (hoặc để trống để dùng tên quốc tế)
TEN_BAO = ""  # Ví dụ: "Bão số 7 (YAGI)"

In [ ]:
# === Phân tích bão ===
import json
import pandas as pd
from typhoon.compare_current_storm import load_current_storm, estimate_landfall, compare_historical

# Tải dữ liệu bão
storm = load_current_storm(geojson_path)
if TEN_BAO:
    storm["storm_name"] = TEN_BAO

# --- Thông tin bão hiện tại ---
pos = storm["current_position"]
ten = storm["storm_name"]
print("=" * 50)
print(f"  THÔNG TIN BÃO: {ten}")
print("=" * 50)
print(f"  Vị trí hiện tại : {pos['lat']:.1f}°N, {pos['lon']:.1f}°E")
print(f"  Thời điểm       : {pos['time_vn']}")
print(f"  Sức gió         : {pos['wind_kt']} kt ({pos['wind_kmh']} km/h)")
print(f"  Phân loại       : {pos['category']}")

# --- Dự báo cực đại ---
if storm["peak_forecast"]:
    pf = storm["peak_forecast"]
    print(f"\n  Dự báo cực đại  : {pf['wind_kt']} kt ({pf['wind_kmh']} km/h)")
    print(f"  Phân loại        : {pf['category']}")
    print(f"  Thời điểm        : {pf['time_vn']} (TAU {pf['tau_h']}h)")

# --- Ước lượng đổ bộ ---
print("\n" + "-" * 50)
print("  ƯỚC LƯỢNG ĐỔ BỘ")
print("-" * 50)
landfall = estimate_landfall(storm)
if landfall and landfall.get("estimated"):
    print(f"  Tỉnh dự kiến    : {landfall['province'] or '(chưa xác định)'}")
    print(f"  Khung thời gian  : {landfall['time_window'][0]} → {landfall['time_window'][1]}")
    print(f"  Sức gió đổ bộ    : {landfall['wind_kt']:.0f} kt ({landfall['wind_kmh']} km/h)")
    print(f"  Phân loại        : {landfall['category']}")
else:
    note = landfall.get("note", "") if landfall else ""
    print(f"  {note or 'Dự báo không cho thấy bão đổ bộ Việt Nam'}")

# --- So sánh lịch sử ---
print("\n" + "-" * 50)
print("  SO SÁNH VỚI LỊCH SỬ")
print("-" * 50)
comparison = compare_historical(storm, landfall)
print(f"  Tổng số bão đổ bộ lịch sử : {comparison['total_historical_landfalls']} ({comparison['year_range']})")
print(f"  Dự báo mạnh hơn           : {comparison['peak_forecast_percentile']}% bão lịch sử")

stats = comparison["historical_wind_stats"]
print(f"  Gió đổ bộ trung bình       : {stats['mean_kph']} km/h")
print(f"  Gió đổ bộ mạnh nhất        : {stats['max_kph']} km/h")

# --- Bảng bão tương tự ---
similar = comparison.get("similar_intensity_storms", [])
if similar:
    print(f"\n  Các bão có cường độ tương tự (±20 km/h):")
    df_similar = pd.DataFrame(similar)
    df_similar = df_similar.rename(columns={
        "NAME": "Tên bão",
        "calc_landfall_time": "Thời gian đổ bộ",
        "wind_at_landfall_kph": "Gió (km/h)",
        "province_landfall": "Tỉnh đổ bộ",
        "time_on_land_h": "Giờ trên đất liền",
    })
    display(df_similar)

# Lưu kết quả
output = {
    "storm": {
        "name": storm["storm_name"],
        "current_position": storm["current_position"],
        "peak_forecast": storm["peak_forecast"],
    },
    "landfall_estimate": landfall,
    "historical_comparison": comparison,
}
out_path = Path("data/current_storm_analysis.json")
out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2))
print(f"\nĐã lưu phân tích: {out_path}")

In [ ]:
# === Bản đồ bão ===
!pip install -q folium
import folium
import json

# Đọc GeoJSON
with open(geojson_path) as f:
    geo = json.load(f)

features = geo["features"]

# Tâm bản đồ = vị trí hiện tại
center_lat = storm["current_position"]["lat"]
center_lon = storm["current_position"]["lon"]
m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")

# Đường đi thực tế (best track line)
for f in features:
    ft = f["properties"]["feature_type"]
    if ft == "best_track":
        coords = [(c[1], c[0]) for c in f["geometry"]["coordinates"]]
        folium.PolyLine(coords, color="#2196F3", weight=2, opacity=0.7,
                        tooltip="Đường đi thực tế").add_to(m)

# Đường dự báo (forecast track line)
for f in features:
    ft = f["properties"]["feature_type"]
    if ft == "forecast_track":
        coords = [(c[1], c[0]) for c in f["geometry"]["coordinates"]]
        folium.PolyLine(coords, color="#F44336", weight=3, opacity=0.8,
                        dash_array="10 5", tooltip="Đường dự báo").add_to(m)

# Vùng nguy hiểm (danger swath)
for f in features:
    ft = f["properties"]["feature_type"]
    if ft == "danger_swath":
        coords = [(c[1], c[0]) for c in f["geometry"]["coordinates"][0]]
        folium.Polygon(coords, color="#FF9800", fill=True, fill_opacity=0.15,
                       weight=1, tooltip="Vùng nguy hiểm (gió >= 34 kt)").add_to(m)

# Các điểm theo dõi (best track points) — xanh dương
for f in features:
    ft = f["properties"]["feature_type"]
    if ft == "best_track_point":
        p = f["properties"]
        lon, lat = f["geometry"]["coordinates"]
        folium.CircleMarker(
            [lat, lon], radius=4, color="#2196F3", fill=True, fill_opacity=0.8,
            tooltip=f"{p['time_vn']} — {p['wind_kmh']} km/h ({p['category']})"
        ).add_to(m)

# Các điểm dự báo (forecast points) — đỏ
for f in features:
    ft = f["properties"]["feature_type"]
    if ft == "forecast_point":
        p = f["properties"]
        lon, lat = f["geometry"]["coordinates"]
        folium.CircleMarker(
            [lat, lon], radius=6, color="#F44336", fill=True, fill_opacity=0.9,
            tooltip=f"TAU {p['tau_h']}h — {p['time_vn']}\n{p['wind_kmh']} km/h ({p['category']})"
        ).add_to(m)
        # Nhãn TAU
        folium.Marker(
            [lat, lon],
            icon=folium.DivIcon(
                html=f'<div style="font-size:10px;color:#D32F2F;font-weight:bold;">{p["tau_h"]}h</div>',
                icon_size=(30, 15), icon_anchor=(0, 0)
            )
        ).add_to(m)

# Đường bờ biển Việt Nam (tham chiếu)
try:
    import geopandas as gpd
    vn = gpd.read_file("data/VietnamMainland.geojson")
    folium.GeoJson(
        vn.to_json(), name="Việt Nam",
        style_function=lambda x: {"color": "#4CAF50", "weight": 2, "fillOpacity": 0.05}
    ).add_to(m)
except Exception:
    pass

# Tiêu đề
title = storm["storm_name"] or "Bão đang hoạt động"
title_html = f'''
<div style="position:fixed;top:10px;left:50px;z-index:9999;
            background:white;padding:8px 16px;border-radius:4px;
            box-shadow:0 2px 6px rgba(0,0,0,0.3);font-size:14px;">
    <b>{title}</b> | <span style="color:#2196F3">Thực tế</span> |
    <span style="color:#F44336">Dự báo</span> |
    <span style="color:#FF9800">Vùng nguy hiểm</span>
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

m

In [ ]:
# === Tải kết quả về máy ===
from google.colab import files
files.download(str(geojson_path))
files.download("data/current_storm_analysis.json")

---

# Cập nhật dữ liệu lịch sử (chỉ chạy khi cần)

> **Phần này dành cho kỹ thuật viên**, không cần chạy khi phân tích bão.

Dữ liệu lịch sử lấy từ IBTrACS (International Best Track Archive). Chỉ cần cập nhật khi:
- Bắt đầu mùa bão mới
- Muốn bổ sung dữ liệu bão gần đây

### Bước 1: Tải IBTrACS mới nhất

```bash
curl -o data/ibtracs.WP.list.v04r01.csv \
  https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.WP.list.v04r01.csv
```

### Bước 2: Chạy pipeline xử lý

```bash
python typhoon/process_landfall.py          # Tính toán đổ bộ
python typhoon/compute_province_metrics.py   # Thống kê theo tỉnh
python typhoon/prepare_dashboard_data.py     # Chuẩn bị dữ liệu dashboard
```

### Bước 3: Commit và push

```bash
git add data/
git commit -m "Cập nhật dữ liệu IBTrACS"
git push
```